In [ ]:
import argparse
import os
import random

import numpy as np
import torch
import torchvision.models as models
from torchvision import transforms

from attack_utils import run_experiment
from impl_myattack import MyAttack


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

datasets = {
    'correct_1000_AlexNet': ('alexnet', models.AlexNet_Weights.IMAGENET1K_V1),
    'correct_1000_DenseNet121': ('densenet121', models.DenseNet121_Weights.IMAGENET1K_V1),
    'correct_1000_GoogLeNet': ('googlenet', models.GoogLeNet_Weights.IMAGENET1K_V1),
    'correct_1000_MobileNetV3_large': ('mobilenet_v3_large', models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    'correct_1000_ResNet34': ('resnet34', models.ResNet34_Weights.IMAGENET1K_V1),
    'correct_1000_VGG11': ('vgg11', models.VGG11_Weights.IMAGENET1K_V1),
    'correct_1000_EfficientNet_b0': ('efficientnet_b0', models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}

config = {
    'SEED': 42,
    'selected_count': 500,
    'output_dir': 'adversarial_samples',
    'attack_name': 'mymodel',
    'MAX_SAVE_ADV': 10,
    'if_save_adv': False,
    'threshold': 1e-6,
    'target_labels': torch.tensor([100]).to(device),
    'if_target': False,
    'if_prune': False,
    # 攻击参数
    'cam_keep_ratio': 0.35, # 有目标攻击时0.45，无目标攻击时0.35
    'eps': 100/255,
    'alpha': 1/255,
    'steps': 100,
    'random_start': False,
    'targeted': False,
    # 剪枝参数
    'step_ratio': 0.01,
    'max_ratio': 1.0,
}

parser = argparse.ArgumentParser(description='Adversarial Attack Experiment')
parser.add_argument('--cam_keep_ratio', type=float, default=0.2, help='CAM keep ratio')
parser.add_argument('--eps', type=float, default=config['eps'], help='Epsilon for MyAttack')
parser.add_argument('--steps', type=int, default=config['steps'], help='Steps for MyAttack')
args = parser.parse_args([])

config.update(vars(args))

SEED = config['SEED']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
if __name__ == "__main__":
    for dataset_name, (model_name, weights) in datasets.items():
        print(f"\n=== 开始实验: {dataset_name} 使用模型 {model_name} ===")

        model = getattr(models, model_name)(weights=weights).to(device)
        model.eval()

        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.ToTensor(),
        ])

        config['val_dir'] = f"./{dataset_name}"
        config['attack_name'] = f"mymodel_{model_name}"

        val_dir = config['val_dir']
        all_images = [f for f in os.listdir(val_dir) if f.endswith('.JPEG')]
        selected_images = random.sample(all_images, min(config['selected_count'], len(all_images)))

        output_dir = config['output_dir']
        attack_name = config['attack_name']
        attack_output_dir = os.path.join(output_dir, attack_name)
        os.makedirs(attack_output_dir, exist_ok=True)
        
        # 循环不同的cam_keep_ratio进行实验
        #all_results = {}
        #for cam_ratio in [0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5]:
        #    config['cam_keep_ratio'] = cam_ratio
        #    all_results[cam_ratio] = run_experiment(config, transform,model, device, selected_images, attack_output_dir, MyAttack)

        results = run_experiment(
            config=config,
            transform=transform,
            model=model,
            device=device,
            selected_images=selected_images,
            attack_output_dir=attack_output_dir,
            attack_cls=MyAttack,
        )

        print(f"=== 实验完成: {dataset_name} ===")
